In [1]:
'''
import os
import numpy as np
import open3d as o3d
import open3d.core as o3c
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

def align_bridge_xy(points):
    """Aligns the point cloud along the X and Y axes using PCA, preserving Z (gravity)."""
    xy_points = points[:, :2]
    pca = PCA(n_components=2)
    pca.fit(xy_points)
    
    # The components give us the rotation matrix for the XY plane
    rot_2d = pca.components_
    
    # Create a 3D rotation matrix (identity for Z)
    rot_3d = np.eye(3)
    rot_3d[:2, :2] = rot_2d
    
    # Apply rotation
    aligned_points = points @ rot_3d.T
    return aligned_points

def prepare_dataset(input_dir, output_dir, target_points=40000):
    os.makedirs(output_dir, exist_ok=True)
    file_list = [f for f in os.listdir(input_dir) if f.endswith('.ply')]
    dataset_records = []

    for file in file_list:
        file_path = os.path.join(input_dir, file)
        
        # Open3D Tensor API loading
        pcd = o3d.t.io.read_point_cloud(file_path)
        points = pcd.point.positions.numpy()
        
        # Extract attributes
        semantic_labels = pcd.point['class_id'].numpy().flatten().astype(np.int64)
        local_instance_labels = pcd.point['instance_id'].numpy().flatten().astype(np.int64)

        # Resample to exactly 40,000 points
        num_points = points.shape[0]
        if num_points > target_points:
            indices = np.random.choice(num_points, target_points, replace=False)
        else:
            indices = np.random.choice(num_points, target_points, replace=True)
            
        points = points[indices]
        semantic_labels = semantic_labels[indices]
        local_instance_labels = local_instance_labels[indices]

        # Map to Global Instance IDs
        global_instance_labels = np.zeros_like(local_instance_labels)
        current_global_id = 0
        
        for class_id in np.unique(semantic_labels):
            class_mask = (semantic_labels == class_id)
            local_ids = np.unique(local_instance_labels[class_mask])
            for local_id in local_ids:
                inst_mask = class_mask & (local_instance_labels == local_id)
                global_instance_labels[inst_mask] = current_global_id
                current_global_id += 1

        # Standardize and Align
        centroid = np.mean(points, axis=0)
        points = points - centroid
        points = align_bridge_xy(points)

        dataset_records.append({
            'points': points,
            'semantic': semantic_labels,
            'instance': global_instance_labels
        })

    # 70/15/15 Split
    train_val, test_data = train_test_split(dataset_records, test_size=0.15, random_state=42)
    train_data, val_data = train_test_split(train_val, test_size=0.1765, random_state=42) 

    np.save(os.path.join(output_dir, 'train_data.npy'), train_data)
    np.save(os.path.join(output_dir, 'val_data.npy'), val_data)
    np.save(os.path.join(output_dir, 'test_data.npy'), test_data)
    print(f"Data split and saved: {len(train_data)} Train, {len(val_data)} Val, {len(test_data)} Test.")
'''


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [1]:
import os
import numpy as np
import open3d as o3d
import open3d.core as o3c
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

def align_bridge_xy(points):
    """Aligns the point cloud along the X and Y axes using PCA, preserving Z (gravity)."""
    xy_points = points[:, :2]
    pca = PCA(n_components=2)
    pca.fit(xy_points)
    
    # The components give us the rotation matrix for the XY plane
    rot_2d = pca.components_
    
    # Create a 3D rotation matrix (identity for Z)
    rot_3d = np.eye(3)
    rot_3d[:2, :2] = rot_2d
    
    # Apply rotation
    aligned_points = points @ rot_3d.T
    return aligned_points

def prepare_dataset(input_dir, output_dir, target_points=40000):
    os.makedirs(output_dir, exist_ok=True)
    file_list = [f for f in os.listdir(input_dir) if f.endswith('.ply')]
    dataset_records = []

    for file in file_list:
        file_path = os.path.join(input_dir, file)
        
        # Open3D Tensor API loading
        pcd = o3d.t.io.read_point_cloud(file_path)
        points = pcd.point.positions.numpy()
        
        # Track raw point count before resampling
        num_actual_points = points.shape[0]
        
        # Extract attributes
        semantic_labels = pcd.point['class_id'].numpy().flatten().astype(np.int64)
        local_instance_labels = pcd.point['instance_id'].numpy().flatten().astype(np.int64)

        # Track unique instance pairs (class_id, instance_id) in the raw cloud
        unique_pairs = set(zip(semantic_labels, local_instance_labels))

        # Resample to exactly 40,000 points
        num_points = points.shape[0]
        if num_points > target_points:
            indices = np.random.choice(num_points, target_points, replace=False)
        else:
            indices = np.random.choice(num_points, target_points, replace=True)
            
        points = points[indices]
        semantic_labels = semantic_labels[indices]
        local_instance_labels = local_instance_labels[indices]

        # Map to Global Instance IDs
        global_instance_labels = np.zeros_like(local_instance_labels)
        current_global_id = 0
        
        for class_id in np.unique(semantic_labels):
            class_mask = (semantic_labels == class_id)
            local_ids = np.unique(local_instance_labels[class_mask])
            for local_id in local_ids:
                inst_mask = class_mask & (local_instance_labels == local_id)
                global_instance_labels[inst_mask] = current_global_id
                current_global_id += 1

        # Standardize and Align
        centroid = np.mean(points, axis=0)
        points = points - centroid
        points = align_bridge_xy(points)

        dataset_records.append({
            'points': points,
            'semantic': semantic_labels,
            'instance': global_instance_labels
        })

        print(f"Processed: {os.path.basename(file_path)} | Raw points: {num_actual_points} | Unique Instances found: {len(unique_pairs)}")

    # 70/15/15 Split
    train_val, test_data = train_test_split(dataset_records, test_size=0.15, random_state=42)
    train_data, val_data = train_test_split(train_val, test_size=0.1765, random_state=42) 

    np.save(os.path.join(output_dir, 'bridge_train.npy'), train_data)
    np.save(os.path.join(output_dir, 'bridge_val.npy'), val_data)
    np.save(os.path.join(output_dir, 'bridge_test.npy'), test_data)
    print(f"\nData split and saved: {len(train_data)} Train, {len(val_data)} Val, {len(test_data)} Test.")



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
if __name__ == "__main__":
    inp_folder = r"/storage/jothi/Yash/Datasets/correct_instance_labeled datasets_with_normals"
    out_folder = r"/storage/jothi/Yash/Datasets/Results_from_133_datasets_modified"
        
    prepare_dataset(inp_folder, out_folder, target_points=40000)

Processed: Bridge CG 01_instance_cloud.ply | Raw points: 40000 | Unique Instances found: 21
Processed: Bridge CG 01_instance_cloud_crop_0.ply | Raw points: 40000 | Unique Instances found: 12
Processed: Bridge CG 01_instance_cloud_crop_1.ply | Raw points: 40000 | Unique Instances found: 15
Processed: Bridge CG 01_instance_cloud_crop_2.ply | Raw points: 40000 | Unique Instances found: 13
Processed: Bridge CG 01_instance_cloud_crop_3.ply | Raw points: 40000 | Unique Instances found: 12
Processed: Bridge CG 02_instance_cloud.ply | Raw points: 40000 | Unique Instances found: 22
Processed: Bridge CG 01_instance_cloud_crop_4.ply | Raw points: 40000 | Unique Instances found: 12
Processed: Bridge CG 02_instance_cloud_crop_0.ply | Raw points: 40000 | Unique Instances found: 9
Processed: Bridge CG 02_instance_cloud_crop_1.ply | Raw points: 40000 | Unique Instances found: 15
Processed: Bridge CG 02_instance_cloud_crop_2.ply | Raw points: 40000 | Unique Instances found: 14
Processed: Bridge CG 02_i

In [4]:
def check_class_distribution(data_list, split_name):
    culvert_count = sum(1 for item in data_list if 6 in np.unique(item['semantic']))
    print(f"{split_name}: {culvert_count} / {len(data_list)} samples contain Culverts")

train_data_list = np.load(os.path.join(out_folder, 'bridge_train.npy'), allow_pickle=True)
val_data_list = np.load(os.path.join(out_folder, 'bridge_val.npy'), allow_pickle=True)
test_data_list = np.load(os.path.join(out_folder, 'bridge_test.npy'), allow_pickle=True)

check_class_distribution(train_data_list, "Train")
check_class_distribution(val_data_list, "Val")
check_class_distribution(test_data_list, "Test")

Train: 4 / 93 samples contain Culverts
Val: 1 / 20 samples contain Culverts
Test: 0 / 20 samples contain Culverts


In [5]:
inp_folder = r"/storage/jothi/Yash/Datasets/correct_instance_labeled datasets_with_normals"
out_folder = r"/storage/jothi/Yash/Datasets/Results_from_dataset_only_culverts_and_cross-girders"

prepare_dataset(inp_folder, out_folder, target_points=40000)

Processed: Bridge CG 01_instance_cloud.ply | Raw points: 40000 | Unique Instances found: 21
Processed: Bridge CG 01_instance_cloud_crop_0.ply | Raw points: 40000 | Unique Instances found: 12
Processed: Bridge CG 01_instance_cloud_crop_1.ply | Raw points: 40000 | Unique Instances found: 15
Processed: Bridge CG 01_instance_cloud_crop_2.ply | Raw points: 40000 | Unique Instances found: 13
Processed: Bridge CG 01_instance_cloud_crop_3.ply | Raw points: 40000 | Unique Instances found: 12
Processed: Bridge CG 02_instance_cloud.ply | Raw points: 40000 | Unique Instances found: 22
Processed: Bridge CG 01_instance_cloud_crop_4.ply | Raw points: 40000 | Unique Instances found: 12
Processed: Bridge CG 02_instance_cloud_crop_0.ply | Raw points: 40000 | Unique Instances found: 9
Processed: Bridge CG 02_instance_cloud_crop_1.ply | Raw points: 40000 | Unique Instances found: 15
Processed: Bridge CG 02_instance_cloud_crop_2.ply | Raw points: 40000 | Unique Instances found: 14
Processed: Bridge CG 02_i

In [6]:
def check_class_distribution(data_list, split_name):
    culvert_count = sum(1 for item in data_list if 6 in np.unique(item['semantic']))
    crossgirder_count = sum(1 for item in data_list if 3 in np.unique(item['semantic']))
    print(f"{split_name}: {culvert_count} / {len(data_list)} samples contain Culverts")
    print(f"{split_name}: {crossgirder_count} / {len(data_list)} samples contain cross-girders")

def verify_split(data_list, name):
    culv_count = sum(1 for item in data_list if 6 in np.unique(item['semantic']))
    cg_count   = sum(1 for item in data_list if 3 in np.unique(item['semantic']))
    print(f"{name} ({len(data_list)} total): {culv_count} Culverts | {cg_count} Cross-girders")

In [8]:
# BALANCING bridge datsets with rare elements like culverts and cross-girders in training, validation and testing datasets

base_dir = r"/storage/jothi/Yash/Datasets/Results_from_dataset_only_culverts_and_cross-girders"

# 1. Load existing splits
train_data = list(np.load(f"{base_dir}/bridge_train.npy", allow_pickle=True))
val_data   = list(np.load(f"{base_dir}/bridge_val.npy", allow_pickle=True))
test_data  = list(np.load(f"{base_dir}/bridge_test.npy", allow_pickle=True))
all_datasets = train_data + val_data + test_data

print("datasets having culvert, and cross-girder classes before modification:")
check_class_distribution(train_data, "Train")
check_class_distribution(val_data, "Val")
check_class_distribution(test_data, "Test")

# 2. Categorize by rare classes
culvert_bridges = []
cross_girder_only = []
standard_bridges = []

for item in all_datasets:
    classes = set(np.unique(item['semantic']))
    if 6 in classes:
        culvert_bridges.append(item)
    elif 3 in classes:
        cross_girder_only.append(item)
    else:
        standard_bridges.append(item)

# 3. Shuffle deterministically
np.random.seed(42)
np.random.shuffle(culvert_bridges)
np.random.shuffle(cross_girder_only)
np.random.shuffle(standard_bridges)

# 4. Stratify Culverts (5 total -> 3 Train, 1 Val, 1 Test)
train_culv = culvert_bridges[:3]
val_culv   = [culvert_bridges[3]]
test_culv  = [culvert_bridges[4]]

# 5. Stratify Cross-girder only bridges (~70% Train, 15% Val, 15% Test)
n_val_cg = int(len(cross_girder_only) * 0.15)
n_test_cg = int(len(cross_girder_only) * 0.15)

val_cg   = cross_girder_only[:n_val_cg]
test_cg  = cross_girder_only[n_val_cg:n_val_cg + n_test_cg]
train_cg = cross_girder_only[n_val_cg + n_test_cg:]

# 6. Stratify remaining standard bridges
n_val_std = int(len(standard_bridges) * 0.15)
n_test_std = int(len(standard_bridges) * 0.15)

val_std   = standard_bridges[:n_val_std]
test_std  = standard_bridges[n_val_std:n_val_std + n_test_std]
train_std = standard_bridges[n_val_std + n_test_std:]

# 7. Merge and save updated splits
final_train = train_culv + train_cg + train_std
final_val   = val_culv + val_cg + val_std
final_test  = test_culv + test_cg + test_std

np.save(f"{base_dir}\\bridge_train_mod.npy", final_train)
np.save(f"{base_dir}\\bridge_val_mod.npy", final_val)
np.save(f"{base_dir}\\bridge_test_mod.npy", final_test)

print("--- Stratified Split Completed ---")

print(f"Train Set ({len(final_train)}): {len(train_culv)} Culverts, {len(train_cg) + len(train_culv)} Cross-girders")
print(f"Val Set   ({len(final_val)}): {len(val_culv)} Culverts, {len(val_cg) + len(val_culv)} Cross-girders")
print(f"Test Set  ({len(final_test)}): {len(test_culv)} Culverts, {len(test_cg) + len(test_culv)} Cross-girders")

datasets having culvert, and cross-girder classes before modification:
Train: 4 / 93 samples contain Culverts
Train: 26 / 93 samples contain cross-girders
Val: 1 / 20 samples contain Culverts
Val: 4 / 20 samples contain cross-girders
Test: 0 / 20 samples contain Culverts
Test: 6 / 20 samples contain cross-girders
--- Stratified Split Completed ---
Train Set (95): 3 Culverts, 29 Cross-girders
Val Set   (19): 1 Culverts, 6 Cross-girders
Test Set  (19): 1 Culverts, 6 Cross-girders


In [9]:
train_data = np.load(f"{base_dir}\\bridge_train_mod.npy", allow_pickle=True)
val_data   = np.load(f"{base_dir}\\bridge_val_mod.npy", allow_pickle=True)
test_data  = np.load(f"{base_dir}\\bridge_test_mod.npy", allow_pickle=True)

verify_split(train_data, "Train")
verify_split(val_data,   "Val")
verify_split(test_data,  "Test")

Train (29 total): 3 Culverts | 26 Cross-girders
Val (6 total): 1 Culverts | 5 Cross-girders
Test (6 total): 1 Culverts | 5 Cross-girders
